In [ ]:
"""
Visualize vessel segmentation results overlaid on the original MRI.

Usage:
    python visualize_segmentation.py                        # auto-picks one file
    python visualize_segmentation.py DUKE_040_0002_vessel_segmentation.npz
    python visualize_segmentation.py --batch                # all completed files

PNGs are saved to visualizations/ inside the vanguard project root by default.
"""

import argparse
import sys
from pathlib import Path

import numpy as np
import SimpleITK as sitk
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


IMAGES_DIR = Path("/scratch/annawoodard/MAMA-MIA-syn60868042/images")
SEG_DIR    = Path("/scratch/scratch1/t-9sarit/vessel_segmentations")
VIZ_DIR    = Path(__file__).resolve().parents[1] / "visualizations"


def find_nii(npz_path: Path) -> Path:
    # e.g. DUKE_040_0002_vessel_segmentation.npz -> DUKE_040_0002.nii.gz
    stem = npz_path.stem.replace("_vessel_segmentation", "")
    case_id = "_".join(stem.split("_")[:-1])  # everything before last _XXXX
    nii = IMAGES_DIR / case_id / f"{stem}.nii.gz"
    if not nii.exists():
        raise FileNotFoundError(f"Could not find MRI at {nii}")
    return nii


def load_mri(nii_path: Path) -> np.ndarray:
    """Load and apply the same axis transform used during preprocessing."""
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(nii_path)))  # (z, y, x)
    arr = np.swapaxes(np.swapaxes(arr, 0, 2), 0, 1)[::-1]       # matches model input space
    return arr.astype(np.float32)


def load_vessel(npz_path: Path) -> np.ndarray:
    return np.load(npz_path)["vessel"].astype(np.float32)


def window(arr: np.ndarray, low=0.5, high=99.5) -> np.ndarray:
    lo, hi = np.percentile(arr, [low, high])
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)


def make_montage(mri: np.ndarray, vessel: np.ndarray, npz_path: Path,
                 threshold: float = 0.3) -> plt.Figure:
    """
    Maximum intensity projection (MIP) along all three axes.
    Rows: MRI MIP | vessel probability MIP | binary vessel MIP
    Columns: axial (z) | coronal (y) | sagittal (x)
    """
    mri_w = window(mri)

    # MIPs: max along each axis
    # mri shape is (Y, X, Z) in transformed space
    projections = [
        ("Axial (z)",    2),   # project along z → view in (Y, X) plane
        ("Coronal (y)",  0),   # project along Y → view in (X, Z) plane
        ("Sagittal (x)", 1),   # project along X → view in (Y, Z) plane
    ]

    fig, axes = plt.subplots(3, 3, figsize=(13, 9), facecolor="black")
    row_labels = ["MRI MIP", "Vessel prob. MIP", f"Binary MIP (>{threshold})"]
    col_labels = [p[0] for p in projections]

    for ax in axes.flat:
        ax.axis("off")
        ax.set_facecolor("black")

    v_global_max = float(vessel.max()) or 1.0

    for c, (col_label, axis) in enumerate(projections):
        mri_mip    = mri_w.max(axis=axis)
        vessel_mip = vessel.max(axis=axis)
        binary_mip = (vessel_mip >= threshold).astype(np.float32)

        axes[0, c].imshow(mri_mip.T, cmap="gray", vmin=0, vmax=1,
                          interpolation="bilinear", aspect="auto")
        axes[0, c].set_title(col_label, color="white", fontsize=10, pad=4)

        axes[1, c].imshow(vessel_mip.T, cmap="hot", vmin=0, vmax=v_global_max,
                          interpolation="bilinear", aspect="auto")

        axes[2, c].imshow(binary_mip.T, cmap="gray", vmin=0, vmax=1,
                          interpolation="nearest", aspect="auto")

    for r, label in enumerate(row_labels):
        axes[r, 0].set_ylabel(label, color="white", fontsize=9)
        axes[r, 0].axis("on")
        axes[r, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        for spine in axes[r, 0].spines.values():
            spine.set_visible(False)

    n_vessel = int((vessel > threshold).sum())
    fig.suptitle(
        f"{npz_path.stem}  |  threshold={threshold}  |  vessel voxels={n_vessel:,}",
        color="white", fontsize=10, y=1.01,
    )
    fig.tight_layout(pad=0.5)
    return fig


def process_one(npz_path: Path, out_path: Path, threshold: float):
    nii_path = find_nii(npz_path)
    print(f"  MRI:    {nii_path}")
    print(f"  vessel: {npz_path}")
    mri    = load_mri(nii_path)
    vessel = load_vessel(npz_path)
    if vessel.shape != mri.shape:
        from skimage.transform import resize
        vessel = resize(vessel, mri.shape, order=1, anti_aliasing=False,
                        preserve_range=True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig = make_montage(mri, vessel, npz_path, threshold=threshold)
    fig.savefig(out_path, dpi=120, bbox_inches="tight", facecolor="black")
    plt.close(fig)
    print(f"  -> {out_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("npz", nargs="?", help="Path to vessel_segmentation.npz")
    parser.add_argument("--batch",     action="store_true", help="Process all completed files")
    parser.add_argument("--threshold", type=float, default=0.3)
    parser.add_argument("--out-dir",   type=Path, default=VIZ_DIR,
                        help=f"Output directory (default: {VIZ_DIR})")
    args = parser.parse_args()

    if args.batch:
        candidates = sorted(SEG_DIR.rglob("*vessel_segmentation.npz"))
        if not candidates:
            sys.exit("No segmentation files found.")
        print(f"Processing {len(candidates)} files -> {args.out_dir}\n")
        for i, npz_path in enumerate(candidates, 1):
            # mirror the dataset/patient subdirectory structure
            rel = npz_path.relative_to(SEG_DIR)
            out_path = args.out_dir / rel.with_suffix(".png")
            if out_path.exists():
                print(f"[{i}/{len(candidates)}] skip (exists): {out_path.name}")
                continue
            print(f"[{i}/{len(candidates)}] {npz_path.name}")
            try:
                process_one(npz_path, out_path, args.threshold)
            except Exception as e:
                print(f"  ERROR: {e}")
    else:
        if args.npz:
            npz_path = Path(args.npz)
        else:
            candidates = sorted(SEG_DIR.rglob("*vessel_segmentation.npz"))
            if not candidates:
                sys.exit("No segmentation files found.")
            npz_path = candidates[0]
            print(f"Auto-selected: {npz_path.name}")

        rel      = npz_path.relative_to(SEG_DIR)
        out_path = args.out_dir / rel.with_suffix(".png")
        process_one(npz_path, out_path, args.threshold)